# HeritageFormer — Heritage Retrieval Engine (clean rebuild v2)

**Goal:** evidence-based restoration reasoning via semantic retrieval — *not* monument classification.

Pipeline: discover datasets -> normalize + clean -> embed with BGE-M3 ->
FAISS search -> knowledge graph -> restoration-reasoning demo.

v2 fixes: strips HTML from descriptions, drops duplicate rows, and lets
restoration search pull evidence from TANGIBLE structures only (so intangible
storytelling rows stop polluting reconstruction evidence).

In [11]:
# ============================================================
# 0) SETUP
# ============================================================
import os, glob, re, pickle, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch:", torch.__version__, "| Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

try:
    import faiss
except ImportError:
    os.system("pip -q install faiss-cpu")
    import faiss
print("FAISS ready")

Torch: 2.10.0+cu128 | Device: cuda
GPU: Tesla T4
FAISS ready


In [12]:
# ============================================================
# 1) AUTO-DISCOVER every CSV under /kaggle/input
#    Mount paths change between sessions -> never hard-code them.
# ============================================================
ALL_CSVS = glob.glob("/kaggle/input/**/*.csv", recursive=True)
print("CSV files found:", len(ALL_CSVS))

def find_csv(*keywords):
    """First CSV whose path contains ALL keywords (case-insensitive)."""
    for p in ALL_CSVS:
        low = p.lower()
        if all(k.lower() in low for k in keywords):
            return p
    return None

paths = {
    "unesco2021": find_csv("tangible", "2021") or find_csv("2021"),
    "unesco2019": find_csv("whc-sites-2019") or find_csv("2019"),
    "heritage3d": find_csv("cultural_heritage_dataset") or find_csv("heritage", "3d"),
    "intangible": find_csv("intangible"),
}
for k, v in paths.items():
    print(f"{k:12s} -> {v}")

CSV files found: 5
unesco2021   -> /kaggle/input/datasets/ramjasmaurya/unesco-heritage-sites2021/whc-sites(tangibles)-2021.csv
unesco2019   -> /kaggle/input/datasets/ujwalkandi/unesco-world-heritage-sites/whc-sites-2019.csv
heritage3d   -> /kaggle/input/datasets/programmer3/high-fidelity-cultural-heritage-3d-dataset/cultural_heritage_dataset.csv
intangible   -> /kaggle/input/datasets/ziya07/cultural-heritage-visual-storytelling-dataset/Intangible_Heritage_Dataset_1983.csv


In [13]:
# ============================================================
# 2) NORMALIZE + CLEAN all datasets into ONE schema
#    name, description, country, region, type, period,
#    material, source, kind, latitude, longitude
#    kind = "tangible" (a real structure) or "intangible"
# ============================================================
TAG_RE = re.compile(r"<[^>]+>")
WS_RE  = re.compile(r"\s+")
def clean(x):
    """Strip HTML tags and collapse whitespace."""
    s = TAG_RE.sub(" ", str(x))
    return WS_RE.sub(" ", s).strip()

def col(df, *cands, default=""):
    for c in cands:
        if c in df.columns:
            return df[c]
    return pd.Series([default] * len(df))

def safe_read(path):
    if not path or not os.path.exists(path):
        return None
    try:
        return pd.read_csv(path)
    except Exception as e:
        print("  ! could not read", path, "|", e)
        return None

frames = []

# --- UNESCO 2021 (tangible sites) ---
df = safe_read(paths["unesco2021"])
if df is not None:
    o = pd.DataFrame()
    o["name"]        = col(df, "Name", "name_en")
    o["description"] = col(df, "short_description", "short_description_en")
    o["country"]     = col(df, "Country name", "states_name_en")
    o["region"]      = col(df, "Region", "region_en")
    o["type"]        = col(df, "category_long", "category")
    o["period"] = ""; o["material"] = ""; o["source"] = "UNESCO2021"; o["kind"] = "tangible"
    o["latitude"]    = col(df, "latitude",  default=np.nan)
    o["longitude"]   = col(df, "longitude", default=np.nan)
    frames.append(o); print("UNESCO2021:", len(o))

# --- UNESCO 2019 (tangible sites) ---
df = safe_read(paths["unesco2019"])
if df is not None:
    o = pd.DataFrame()
    o["name"]        = col(df, "name_en", "Name")
    o["description"] = col(df, "short_description_en", "short_description")
    o["country"]     = col(df, "states_name_en", "Country name")
    o["region"]      = col(df, "region_en", "Region")
    o["type"]        = col(df, "category", "category_long")
    o["period"] = ""; o["material"] = ""; o["source"] = "UNESCO2019"; o["kind"] = "tangible"
    o["latitude"]    = col(df, "latitude",  default=np.nan)
    o["longitude"]   = col(df, "longitude", default=np.nan)
    frames.append(o); print("UNESCO2019:", len(o))

# --- High-Fidelity Cultural Heritage 3D ---
df = safe_read(paths["heritage3d"])
if df is not None:
    o = pd.DataFrame()
    o["name"]        = col(df, "Artifact_Name", "name")
    o["description"] = ("3D scanned heritage artifact. Scan type "
                        + col(df, "Scan_Type").astype(str)
                        + ", resolution " + col(df, "Resolution").astype(str)
                        + ", level of detail " + col(df, "LOD_Level").astype(str) + ".")
    o["country"] = ""; o["region"] = ""
    o["type"]        = "3D Artifact"
    o["period"]      = col(df, "Historical_Period")
    o["material"] = ""; o["source"] = "Heritage3D"; o["kind"] = "tangible"
    o["latitude"] = np.nan; o["longitude"] = np.nan
    frames.append(o); print("Heritage3D:", len(o))

# --- Intangible heritage (visual storytelling) ---
df = safe_read(paths["intangible"])
if df is not None:
    o = pd.DataFrame()
    o["name"]        = col(df, "Heritage_Type", "File_Name", "ID").astype(str)
    o["description"] = col(df, "Description")
    o["country"] = ""
    o["region"]      = col(df, "Region")
    o["type"]        = "Intangible: " + col(df, "Narrative_Label", "Modality").astype(str)
    o["period"] = ""; o["material"] = ""; o["source"] = "Intangible"; o["kind"] = "intangible"
    o["latitude"] = np.nan; o["longitude"] = np.nan
    frames.append(o); print("Intangible:", len(o))

heritage_master = pd.concat(frames, ignore_index=True)

# clean text fields (removes <p> HTML, collapses whitespace)
for c in ["name","description","country","region","type","period","material"]:
    heritage_master[c] = heritage_master[c].fillna("").map(clean)

# drop blanks and exact-duplicate records (kills the "Temple Architecture" spam)
heritage_master = heritage_master[heritage_master["name"] != ""]
heritage_master = heritage_master.drop_duplicates(
    subset=["name","type","region","description"]).reset_index(drop=True)

print("\nHERITAGE MASTER:", heritage_master.shape)
print(heritage_master.groupby(["kind","source"]).size())
heritage_master.head()

UNESCO2021: 1155
UNESCO2019: 1121
Heritage3D: 500
Intangible: 1983

HERITAGE MASTER: (3548, 11)
kind        source    
intangible  Intangible    1814
tangible    Heritage3D     452
            UNESCO2019     127
            UNESCO2021    1155
dtype: int64


,name,description,country,region,type,period,material,source,kind,latitude,longitude
0,L’Anse aux Meadows National Historic Site,At the tip of the Great Northern Peninsula of ...,Canada,Europe and North America,Cultural,,,UNESCO2021,tangible,51.466667,-55.616667
1,Nahanni National Park,"Located along the South Nahanni River, one of ...",Canada,Europe and North America,Natural,,,UNESCO2021,tangible,61.547222,-125.589444
2,Galápagos Islands,"Situated in the Pacific Ocean some 1,000 km fr...",Ecuador,Latin America and the Caribbean,Natural,,,UNESCO2021,tangible,-0.689860,-90.501319
3,City of Quito,"Quito, the capital of Ecuador, was founded in ...",Ecuador,Latin America and the Caribbean,Cultural,,,UNESCO2021,tangible,-0.220000,-78.512083
4,Simien National Park,Massive erosion over the years on the Ethiopia...,Ethiopia,Africa,Natural,,,UNESCO2021,tangible,13.183333,38.066667


In [14]:
# ============================================================
# 3) Build a rich "Heritage Knowledge Record" per row.
#    Richer text -> stronger BGE-M3 embeddings -> better retrieval.
# ============================================================
def build_record(r):
    parts = [f"Heritage Name: {r['name']}"]
    if r["type"]:        parts.append(f"Heritage Type: {r['type']}")
    if r["period"]:      parts.append(f"Historical Period: {r['period']}")
    if r["country"]:     parts.append(f"Country: {r['country']}")
    if r["region"]:      parts.append(f"Region: {r['region']}")
    if r["material"]:    parts.append(f"Construction Material: {r['material']}")
    if r["description"]: parts.append(f"Description: {r['description']}")
    parts.append("Context: cultural, architectural and archaeological reference "
                 "record used for evidence-based restoration and reconstruction.")
    return "\n".join(parts)

heritage_master["record_text"] = [build_record(r) for _, r in heritage_master.iterrows()]
print(heritage_master["record_text"].iloc[0][:400])

Heritage Name: L’Anse aux Meadows National Historic Site
Heritage Type: Cultural
Country: Canada
Region: Europe and North America
Description: At the tip of the Great Northern Peninsula of the island of Newfoundland, the remains of an 11th-century Viking settlement are evidence of the first European presence in North America. The excavated remains of wood-framed peat-turf buildings are similar to 


In [15]:
# ============================================================
# 4) Embed every record with BGE-M3 (1024-dim, multilingual)
# ============================================================
from sentence_transformers import SentenceTransformer
encoder = SentenceTransformer("BAAI/bge-m3", device=DEVICE)

records = heritage_master["record_text"].tolist()
embeddings = encoder.encode(
    records,
    batch_size=64,
    normalize_embeddings=True,   # required for cosine / inner-product search
    show_progress_bar=True,
    convert_to_numpy=True,
).astype("float32")

np.save("/kaggle/working/embeddings.npy", embeddings)
print("Embeddings:", embeddings.shape)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/56 [00:00<?, ?it/s]

Embeddings: (3548, 1024)


In [16]:
# ============================================================
# 5) FAISS index (inner product == cosine on normalized vectors)
# ============================================================
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)
faiss.write_index(index, "/kaggle/working/heritage.index")
print("Indexed vectors:", index.ntotal)

Indexed vectors: 3548


In [17]:
# ============================================================
# 6) Search engine + restoration-oriented query builder
#    tangible_only=True -> evidence comes from real structures
#    (UNESCO + Heritage3D), excluding intangible storytelling rows.
# ============================================================
def heritage_search(query, top_k=10, source=None, tangible_only=False):
    q = encoder.encode([query], normalize_embeddings=True).astype("float32")
    over = 8 if (source or tangible_only) else 1     # over-fetch then filter
    scores, idx = index.search(q, top_k * over)
    res = heritage_master.iloc[idx[0]].copy()
    res["score"] = scores[0]
    if tangible_only:
        res = res[res["kind"] == "tangible"]
    if source:
        res = res[res["source"] == source]
    cols = ["name","country","region","type","period","source","score"]
    return res.head(top_k)[cols].reset_index(drop=True)

def restoration_query(structure_type, period="", material="", location=""):
    return (f"{period} {structure_type} {material} {location} "
            "historical architecture defensive architecture "
            "heritage conservation restoration reference").strip()

heritage_search(restoration_query("hill fort", "17th century maratha",
                                  "basalt stone", "India"),
                tangible_only=True)

,name,country,region,type,period,source,score
0,Hill Forts of Rajasthan,India,Asia and the Pacific,Cultural,,UNESCO2021,0.594307
1,Agra Fort,India,Asia and the Pacific,Cultural,,UNESCO2021,0.572535
2,Bahla Fort,Oman,Arab States,Cultural,,UNESCO2021,0.564873
3,Brimstone Hill Fortress National Park,Saint Kitts and Nevis,Latin America and the Caribbean,Cultural,,UNESCO2021,0.549095


In [18]:
# ============================================================
# 7) Knowledge graph (NetworkX) — richer node & edge types,
#    plus analogical SIMILAR_TO edges mined from retrieval.
# ============================================================
import networkx as nx
G = nx.MultiDiGraph()

def add_node(n, t):
    n = str(n).strip()
    if n:
        G.add_node(n, node_type=t)

for _, r in heritage_master.iterrows():
    site = str(r["name"]).strip()
    add_node(site, "site")
    for val, typ, rel in [
        (r["country"],  "country",  "LOCATED_IN"),
        (r["region"],   "region",   "IN_REGION"),
        (r["type"],     "type",     "HAS_TYPE"),
        (r["period"],   "period",   "FROM_PERIOD"),
        (r["material"], "material", "USES_MATERIAL"),
        (r["source"],   "source",   "DOCUMENTED_IN"),
    ]:
        if str(val).strip():
            add_node(val, typ)
            G.add_edge(site, str(val).strip(), relation=rel)

# analogical edges: link each site to its 3 nearest neighbours
D, I = index.search(embeddings, 4)          # 4 = self + 3 neighbours
for i in range(len(heritage_master)):
    src = str(heritage_master.iloc[i]["name"]).strip()
    for j, sc in zip(I[i][1:], D[i][1:]):   # skip self (column 0)
        dst = str(heritage_master.iloc[j]["name"]).strip()
        if src != dst:
            G.add_edge(src, dst, relation="SIMILAR_TO", weight=float(sc))

with open("/kaggle/working/heritage_graph.pkl", "wb") as f:
    pickle.dump(G, f)

types = {}
for _, d in G.nodes(data=True):
    t = d.get("node_type", "?"); types[t] = types.get(t, 0) + 1
print("Nodes:", G.number_of_nodes(), "| Edges:", G.number_of_edges())
print("Node types:", types)

Nodes: 1472 | Edges: 15557
Node types: {'site': 1220, 'country': 205, 'region': 19, 'type': 14, 'source': 4, 'period': 10}


In [19]:
# ============================================================
# 8) RESTORATION REASONING — worked example
#    Konark Sun Temple: its main tower (deul) collapsed long ago.
#    HeritageFormer reasons by analogy to INTACT similar temples.
# ============================================================
def restoration_report(target_hint, structure_type, period, material, location):
    print("=" * 64)
    print("RESTORATION TARGET:", target_hint)
    print("=" * 64)

    # resolve the closest real (tangible) record for the target
    match = heritage_search(target_hint, top_k=1, tangible_only=True)
    target = match.iloc[0]["name"] if len(match) else target_hint
    print("Resolved to record:", target, "\n")

    # evidence pool = real structures only
    refs = heritage_search(restoration_query(structure_type, period, material, location),
                           top_k=8, tangible_only=True)
    print("Top reference structures (tangible evidence pool):\n")
    print(refs.to_string(index=False))

    print("\nClosest analogues (graph SIMILAR_TO edges):")
    if target in G:
        sims = [(v, d["weight"]) for _, v, d in G.out_edges(target, data=True)
                if d.get("relation") == "SIMILAR_TO"]
        for v, w in sorted(sims, key=lambda x: -x[1])[:5]:
            print(f"   - {v}  (similarity {w:.3f})")
    else:
        print("   (target not yet a graph node)")

    print("\nReasoning trace:")
    print(" 1. Retrieve historically/structurally similar real structures (above).")
    print(" 2. Keep only same type / period / material as supporting evidence.")
    print(" 3. Infer missing components from the INTACT analogues.")
    print(" 4. Justify every reconstruction choice with its source record.")

restoration_report(
    target_hint="Konark Sun Temple Odisha",
    structure_type="Kalinga style Hindu temple shikhara tower",
    period="13th century Eastern Ganga dynasty",
    material="khondalite sandstone",
    location="Odisha India",
)

RESTORATION TARGET: Konark Sun Temple Odisha
Resolved to record: Sun Temple, Konârak 

Top reference structures (tangible evidence pool):

Empty DataFrame
Columns: [name, country, region, type, period, source, score]
Index: []

Closest analogues (graph SIMILAR_TO edges):
   - Buddhist Monuments at Sanchi  (similarity 0.610)
   - Ellora Caves  (similarity 0.605)
   - Bagan  (similarity 0.597)

Reasoning trace:
 1. Retrieve historically/structurally similar real structures (above).
 2. Keep only same type / period / material as supporting evidence.
 3. Infer missing components from the INTACT analogues.
 4. Justify every reconstruction choice with its source record.


In [20]:
# ============================================================
# 9) Save artifacts for reuse / next stages
# ============================================================
heritage_master.to_parquet("/kaggle/working/heritage_master.parquet")
print("Saved to /kaggle/working:")
for f in ["heritage_master.parquet", "embeddings.npy",
          "heritage.index", "heritage_graph.pkl"]:
    p = f"/kaggle/working/{f}"
    if os.path.exists(p):
        print(f"  {f:28s} {os.path.getsize(p)/1e6:7.1f} MB")

Saved to /kaggle/working:
  heritage_master.parquet          1.1 MB
  embeddings.npy                  14.5 MB
  heritage.index                  14.5 MB
  heritage_graph.pkl               0.6 MB


---
# v3 — Component-Level Restoration Reasoning

The v2 engine retrieves similar *sites* but can't reason about missing *parts*
(Konark's collapsed tower, a fort's lost bastion). v3 adds:

1. A hand-authored **component ontology** (canonical parts per structure archetype) — citeable, no fictional data.
2. **Missing-component inference**: for a target structure, list its canonical
   components, flag documented losses, and retrieve INTACT exemplars of each
   missing component from analogous sites — with an evidence trail + confidence.

No GPU and no training — this runs on the retrieval + graph you already built.

In [21]:
# ============================================================
# 10) COMPONENT ONTOLOGY  (hand-authored, citeable)
#     Canonical components per structure archetype.
# ============================================================
component_ontology = {
    "kalinga_temple": [
        "deul / rekha shikhara (curvilinear main tower)",
        "jagamohana (assembly hall / mandapa)",
        "nata-mandira (dance hall)",
        "bhoga-mandapa (hall of offerings)",
        "amalaka and kalasha (crowning finial)",
    ],
    "nagara_temple": [
        "shikhara (curvilinear spire)",
        "garbhagriha (sanctum)",
        "mandapa (pillared hall)",
        "antarala (vestibule)",
        "amalaka and kalasha (finial)",
    ],
    "dravidian_temple": [
        "vimana (pyramidal tower over sanctum)",
        "gopuram (gateway tower)",
        "mandapa (pillared hall)",
        "garbhagriha (sanctum)",
        "prakara (enclosure wall)",
    ],
    "mughal_tomb": [
        "double dome",
        "char-bagh (formal garden)",
        "pishtaq / iwan (recessed portal)",
        "minaret",
        "raised plinth",
    ],
    "indo_islamic": [
        "dome",
        "minaret",
        "pointed arch",
        "courtyard (sahn)",
        "iwan (portal)",
    ],
    "hill_fort": [
        "rampart (defensive wall)",
        "bastion (burj)",
        "gateway (darwaza)",
        "watchtower",
        "water cistern / tanka",
        "citadel (inner fort)",
    ],
    "cave_temple": [
        "chaitya hall (prayer hall)",
        "vihara (monastic cells)",
        "rock-cut facade",
        "rock-cut pillars",
        "stupa / shrine",
    ],
    "buddhist_monument": [
        "stupa (hemispherical dome)",
        "harmika and chhatra (crowning umbrella)",
        "toranas (gateways)",
        "circumambulatory path (pradakshina)",
        "vedika (railing)",
    ],
    "generic_monument": [
        "foundation / plinth",
        "load-bearing walls",
        "superstructure / roof",
        "entrance / gateway",
        "ornamentation",
    ],
}

# short retrieval descriptor per archetype (used to find intact exemplars)
ARCHETYPE_DESC = {
    "kalinga_temple":  "Kalinga Odisha Hindu temple",
    "nagara_temple":   "Nagara north Indian Hindu temple",
    "dravidian_temple":"Dravidian south Indian Hindu temple",
    "mughal_tomb":     "Mughal Indo-Islamic tomb mausoleum",
    "indo_islamic":    "Indo-Islamic mosque monument",
    "hill_fort":       "Indian hill fort defensive architecture",
    "cave_temple":     "Indian rock-cut cave temple",
    "buddhist_monument":"Buddhist stupa monument",
    "generic_monument":"historic monument",
}

def infer_archetype(name, typ, desc, region):
    """Map a record to a structure archetype using keyword rules."""
    t = f"{name} {typ} {desc} {region}".lower()
    def has(*ks): return any(k in t for k in ks)
    if has("konâ", "konar", "kalinga") or (has("temple") and "odisha" in t):
        return "kalinga_temple"
    if has("chola", "vimana", "gopuram", "dravid", "mahabalipuram", "mamallapuram", "pallava"):
        return "dravidian_temple"
    if has("khajuraho") or (has("temple") and has("nagara")):
        return "nagara_temple"
    if has("tomb", "mausoleum", "maqbara"):
        return "mughal_tomb"
    if has("stupa") or has("sanchi") or has("buddhist monument"):
        return "buddhist_monument"
    if has("cave", "ajanta", "ellora", "elephanta"):
        return "cave_temple"
    if has("minar", "minaret", "mosque", "masjid", "qutb"):
        return "indo_islamic"
    if has("fort", "fortress", "qila", "durg", "rampart"):
        return "hill_fort"
    if has("temple"):
        return "nagara_temple"
    return "generic_monument"

# documented losses for well-studied sites (curated; cite ASI/UNESCO)
KNOWN_MISSING = {
    "Sun Temple, Konârak": {
        "deul / rekha shikhara (curvilinear main tower)":
            "collapsed by c.1837; once ~70 m and the tallest element, now lost",
    },
    "Group of Monuments at Hampi": {
        "superstructure / roof":
            "extensive ruin after 1565 sack of Vijayanagara; many superstructures lost",
    },
}
print("Ontology archetypes:", list(component_ontology))

Ontology archetypes: ['kalinga_temple', 'nagara_temple', 'dravidian_temple', 'mughal_tomb', 'indo_islamic', 'hill_fort', 'cave_temple', 'buddhist_monument', 'generic_monument']


In [22]:
# ============================================================
# 11) MISSING-COMPONENT INFERENCE + worked restoration plan
# ============================================================
def _confidence(score):
    if score >= 0.55: return "HIGH"
    if score >= 0.45: return "MEDIUM"
    return "LOW"

def component_evidence(component, archetype, exclude_name, k=3):
    """Retrieve intact tangible exemplars that best evidence a component."""
    q = f"{ARCHETYPE_DESC.get(archetype,'')} {component} intact complete architecture India"
    r = heritage_search(q, top_k=k + 3, tangible_only=True)
    r = r[r["name"] != exclude_name].head(k)
    return r

def restoration_plan(target_hint, location="India"):
    print("=" * 70)
    print("RESTORATION PLAN:", target_hint)
    print("=" * 70)

    match = heritage_search(f"{target_hint} {location}", top_k=1, tangible_only=True)
    if not len(match):
        print("No tangible record found."); return
    row = match.iloc[0]
    target = row["name"]

    # recover full record (archetype needs the description)
    rec = heritage_master[heritage_master["name"] == target].iloc[0]
    arch = infer_archetype(rec["name"], rec["type"], rec["description"], rec["region"])
    canon = component_ontology[arch]

    # documented losses for this target (substring match, accents safe)
    missing = {}
    for k, v in KNOWN_MISSING.items():
        if k.lower() in target.lower() or target.lower() in k.lower():
            missing = v
    missing_set = set(missing)

    print(f"Resolved record : {target}")
    print(f"Archetype       : {arch}")
    print(f"Canonical parts : {len(canon)}\n")

    print("COMPONENT STATUS")
    print("-" * 70)
    for c in canon:
        tag = "MISSING" if c in missing_set else "present / to verify"
        note = f"  <- {missing[c]}" if c in missing_set else ""
        print(f"  [{tag:18s}] {c}{note}")

    if not missing_set:
        print("\nNo documented losses for this site. Showing reference exemplars "
              "for every canonical component instead.")
    targets = list(missing_set) if missing_set else canon

    print("\nEVIDENCE-BASED RECONSTRUCTION REFERENCES")
    print("-" * 70)
    for c in targets:
        ev = component_evidence(c, arch, target)
        print(f"\n  Component: {c}")
        if not len(ev):
            print("    (no analogous exemplar found)"); continue
        for _, e in ev.iterrows():
            print(f"    - {e['name']}  [{e['source']}]  "
                  f"sim={e['score']:.3f}  confidence={_confidence(e['score'])}")

    print("\nINFERENCE CHAIN")
    print("-" * 70)
    print(" 1. Classify target -> archetype -> canonical component set.")
    print(" 2. Flag components with documented loss (curated from ASI/UNESCO).")
    print(" 3. For each missing part, retrieve INTACT exemplars of the SAME")
    print("    archetype as reconstruction evidence (tangible records only).")
    print(" 4. Confidence = retrieval similarity (style+period+region proximity).")
    print(" 5. Every suggested form is traceable to a cited source record.")
    print("\nNOTE: this proposes evidence-backed references, not generated imagery.")

restoration_plan("Konark Sun Temple Odisha")

RESTORATION PLAN: Konark Sun Temple Odisha
Resolved record : Sun Temple, Konârak
Archetype       : kalinga_temple
Canonical parts : 5

COMPONENT STATUS
----------------------------------------------------------------------
  [MISSING           ] deul / rekha shikhara (curvilinear main tower)  <- collapsed by c.1837; once ~70 m and the tallest element, now lost
  [present / to verify] jagamohana (assembly hall / mandapa)
  [present / to verify] nata-mandira (dance hall)
  [present / to verify] bhoga-mandapa (hall of offerings)
  [present / to verify] amalaka and kalasha (crowning finial)

EVIDENCE-BASED RECONSTRUCTION REFERENCES
----------------------------------------------------------------------

  Component: deul / rekha shikhara (curvilinear main tower)
    (no analogous exemplar found)

INFERENCE CHAIN
----------------------------------------------------------------------
 1. Classify target -> archetype -> canonical component set.
 2. Flag components with documented loss (curated

---
# Evaluation Harness — measure retrieval quality

Eyeballing results isn't enough for publication or for proving v3 > v2. This
scores the engine on a small labelled set of restoration-style queries using
**Recall@5, Recall@10 and MRR** (mean reciprocal rank). A query "hits" if any
accepted site name appears in the top-k tangible results.

In [23]:
# ============================================================
# 12) RETRIEVAL EVAL — Recall@k and MRR on labelled queries
# ============================================================
# (query, [accepted name substrings]) — all are real UNESCO India sites
EVAL_SET = [
    ("mughal tomb white marble dome garden",          ["taj mahal"]),
    ("mughal emperor tomb delhi garden",              ["humayun"]),
    ("rajasthan hill fort defensive walls",           ["hill forts of rajasthan"]),
    ("mughal red sandstone fort delhi",               ["red fort"]),
    ("mughal fort agra",                              ["agra fort"]),
    ("buddhist rock cut cave paintings",              ["ajanta"]),
    ("rock cut cave temples kailasa",                 ["ellora"]),
    ("sun temple chariot wheels odisha",              ["sun temple", "konâr", "konar"]),
    ("mountain railway india heritage",               ["mountain railways"]),
    ("vijayanagara ruins hampi temples",              ["hampi"]),
    ("great buddhist stupa sanchi",                   ["sanchi"]),
    ("khajuraho temples sculpture",                   ["khajuraho"]),
    ("great living chola temples south india",        ["chola"]),
    ("qutb minar tower delhi",                        ["qutb"]),
    ("fatehpur sikri mughal city",                    ["fatehpur sikri"]),
    ("churches convents goa portuguese",              ["goa"]),
    ("pallava monuments mahabalipuram shore temple",  ["mahabalipuram", "mamallapuram"]),
    ("elephanta island cave shiva",                   ["elephanta"]),
]

def evaluate(eval_set, k_values=(5, 10)):
    maxk = max(k_values)
    hits = {k: 0 for k in k_values}
    rr_sum = 0.0
    misses = []
    for query, gold in eval_set:
        res = heritage_search(query, top_k=maxk, tangible_only=True)
        names = [n.lower() for n in res["name"].tolist()]
        rank = None
        for i, nm in enumerate(names):
            if any(g in nm for g in gold):
                rank = i + 1; break
        for k in k_values:
            if rank is not None and rank <= k:
                hits[k] += 1
        rr_sum += (1.0 / rank) if rank else 0.0
        if rank is None:
            misses.append((query, gold))
    n = len(eval_set)
    print("RETRIEVAL QUALITY  (tangible_only, n =", n, ")")
    print("-" * 50)
    for k in k_values:
        print(f"  Recall@{k:<2d} : {hits[k]/n:.3f}  ({hits[k]}/{n})")
    print(f"  MRR      : {rr_sum/n:.3f}")
    if misses:
        print("\n  Misses (not found in top", maxk, "):")
        for q, g in misses:
            print(f"    - '{q}'  expected {g}")
    return {"recall": {k: hits[k]/n for k in k_values}, "mrr": rr_sum/n}

eval_metrics = evaluate(EVAL_SET)

RETRIEVAL QUALITY  (tangible_only, n = 18 )
--------------------------------------------------
  Recall@5  : 1.000  (18/18)
  Recall@10 : 1.000  (18/18)
  MRR      : 0.935


---
# Metrics Logging — version history

Every eval run is appended to `metrics_history.json` (+ `.csv`) in
`/kaggle/working`, so Recall@5 / Recall@10 / MRR are tracked **across versions**.

Bump the `version=` label whenever you change something (more records, hybrid
retrieval, reranking…). Re-running the same version label overwrites its row
instead of duplicating. To keep history across Kaggle sessions, download the
JSON (or save it as a dataset) and re-attach it next run — the loader picks it
up automatically from any attached input.

In [24]:
# ============================================================
# 13) METRICS LOGGING — track Recall/MRR across versions
# ============================================================
import json as _json, datetime
import pandas as pd

METRICS_PATH = "/kaggle/working/metrics_history.json"
METRICS_CSV  = "/kaggle/working/metrics_history.csv"

def load_history():
    """Load existing history from working dir, else from any attached input."""
    candidates = [METRICS_PATH] + glob.glob(
        "/kaggle/input/**/metrics_history.json", recursive=True)
    for p in candidates:
        if os.path.exists(p):
            try:
                return _json.load(open(p))
            except Exception:
                pass
    return []

def log_metrics(version, metrics, n_records, notes="", replace_same_version=True):
    hist = load_history()
    if replace_same_version:
        hist = [r for r in hist if r.get("version") != version]
    row = {
        "version":   version,
        "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
        "n_records": int(n_records),
        "recall@5":  round(metrics["recall"].get(5,  float("nan")), 4),
        "recall@10": round(metrics["recall"].get(10, float("nan")), 4),
        "mrr":       round(metrics["mrr"], 4),
        "encoder":   "BAAI/bge-m3",
        "notes":     notes,
    }
    hist.append(row)
    _json.dump(hist, open(METRICS_PATH, "w"), indent=2)
    df = pd.DataFrame(hist)
    df.to_csv(METRICS_CSV, index=False)
    return df

history_df = log_metrics(
    version="v3-dense-tangible",
    metrics=eval_metrics,
    n_records=len(heritage_master),
    notes="BGE-M3 dense + FAISS IP, tangible-only eval, deduped+cleaned records",
)

print("METRICS HISTORY")
print("=" * 70)
print(history_df[["version","n_records","recall@5","recall@10","mrr","timestamp"]]
      .to_string(index=False))

# improvement vs the previous DIFFERENT version, if any
prev = history_df[history_df["version"] != history_df.iloc[-1]["version"]]
if len(prev):
    p, c = prev.iloc[-1], history_df.iloc[-1]
    print(f"\nvs {p['version']}:  "
          f"Recall@10 {c['recall@10']-p['recall@10']:+.4f}   "
          f"MRR {c['mrr']-p['mrr']:+.4f}")
else:
    print("\n(first logged run — this is your baseline)")

METRICS HISTORY
          version  n_records  recall@5  recall@10    mrr           timestamp
v3-dense-tangible       3548       1.0        1.0 0.9352 2026-06-23T01:57:26

(first logged run — this is your baseline)
